# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Beshair-Khan/flyrank_ml_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os
from dotenv import load_dotenv
import pandas as pd
load_dotenv()
hf_token=os.getenv('HF_TOKEN')
print("Token added successfully ", hf_token is not None)
df_c=pd.read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet', storage_options={'token': hf_token})
df_f=pd.read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet', storage_options={'token':hf_token})
print(df_c.shape, df_f.shape)
print(df_f.columns.tolist())
print(df_f['content_id'].nunique() if 'content_id' in df_f.columns else 'check id column name')

Token added successfully  True
(519606, 26) (9841378, 31)
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
check id column name


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* **Unit Of Analysis:** 1 unique row=1 unique webpage identified by content_id
* **Time Windows:** We use march 2026 mid panel window
* **Table Used:** content_refresh_anonymized.csv
* **Predict / Rank:** opportunity_flags (0-3 count of risk conditions: declining, page-one, stale) a transparent baseline score, not a weighted formula.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.

 **Features**
 * search_volume
 * cpc
 * avg_position
 * word_count
 * trend_pct
 * Features are knowable at the decision moment (March 2026) because search volume, rankings, word count, and trend slopes are logged historical metrics from Search Console/Analytics prior to making refresh decisions.

 **Label**
 * opportunity_flags: count of true risk conditions (declining, page-one, stale) transparent, no invented weights

 **Context**
 * content_id: Unique anonymized webpage identifier.
 * main_intent: Categorical intent (informational, transactional, commercial).

 **Excluded**
 * url, client_name, domain: Excluded for privacy compliance (anonymization).
 * Future traffic/rankings: Excluded to avoid temporal data leakage (using future outcome states to predict current priority).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df['is_declining']=(df['trend_direction']=='down').astype(int)
df['is_page_one']=(df['position_tier']=='page_1').astype(int)
df['is_high_value']=(df['cpc']>df['cpc'].median()).astype(int)
df['opportunity_score'] = ((df['is_declining'] * 35) +(df['is_page_one'] * 25) +(df['is_high_value'] * 20) +((1 - df['ctr'].fillna(0)) * 20)).round(1)
features = ['search_volume', 'avg_position', 'cpc', 'word_count', 'trend_pct']
label = ['opportunity_score']
context = ['content_id', 'client_id', 'main_intent']
print("Field Buckets Assigned Successfully:")
print(f"Features ({len(features)}): {features}")
print(f"Label (1): {label}")
print(f"Context ({len(context)}): {context}")

Field Buckets Assigned Successfully:
Features (5): ['search_volume', 'avg_position', 'cpc', 'word_count', 'trend_pct']
Label (1): ['opportunity_score']
Context (3): ['content_id', 'client_id', 'main_intent']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Time Window: This starter snapshot (content_refresh_anonymized.csv) has no date/month column — it's a single static cross-section, not time-partitioned. A real time window (e.g. month=2026-03) will apply once we move to the Hugging Face warehouse release in Week 3.

* **Grain Check:** Confirming zero duplicate content_id entries.
* **Row Count & Completeness:** Measuring overall row count and non-null availability across features.
* **Availability Filter (IS TRUE check):** Verifying how many rows survive when requiring valid, non-null feature values.

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
dup=df.duplicated(subset=['content_id']).sum()
print(f"The duplicate grains are {dup}")
print(f"\nThe total rows are {len(df):,}")
print("\nChecking null values in features")
print(df[features].notnull().sum())
is_complete_row = df[features].notnull().all(axis=1) & df['content_id'].notnull()
surviving_rows = is_complete_row.sum()
survival_pct = (surviving_rows / total_rows) * 100
print("\nAVAILABILITY FILTER (IS TRUE)")
print(f"Feature columns checked: {features}")
print(f"Surviving rows after filter: {surviving_rows:,} / {total_rows:,} ({survival_pct:.1f}% survived)")

The duplicate grains are 0

The total rows are 30,000

Checking null values in features
search_volume    27532
avg_position     30000
cpc              27532
word_count       22301
trend_pct        26612
dtype: int64

AVAILABILITY FILTER (IS TRUE)
Feature columns checked: ['search_volume', 'avg_position', 'cpc', 'word_count', 'trend_pct']
Surviving rows after filter: 18,013 / 30,000 (60.0% survived)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What can this data never tell you?**

This dataset cannot observe external search engine algorithm updates or competitor backlink changes. A drop in traffic might be driven by external core updates rather than content staleness.

In [44]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df['cheat'] = df['opportunity_score'] * 0.99
print("Cheat Correlation:", df['cheat'].corr(df['opportunity_score']))
df.drop(columns=['cheat'], inplace=True)
print("cheat column is removed")

Cheat Correlation: 1.0
cheat column is removed


## Self-check

Before you submit, confirm each line honestly:

- [ Yes ] Every section above is filled — markdown thinking AND the code that backs it
- [ Yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ Yes ] No client names, URLs, or private queries anywhere
- [ Yes ] My claims use careful words: observed, measured, directional, decision-support
- [ Yes ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.